In [51]:
from dotenv import load_dotenv , find_dotenv
import os

#load dot env
load_dotenv(find_dotenv())

#initialise the api keys variables
narott
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_ENDPOINT")

In [67]:
#initialising and importing important langcahin function and utilities
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_classic.agents import AgentExecutor, create_react_agent, initialize_agent
from langchain_classic import hub

#initialsing the LLM Memory
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

#importing important tools and utilities
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_experimental.tools import PythonREPLTool

In [68]:
#initialising the llm
llm = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model= "azure.gpt-4.1-mini"
)

#initialising the cache
set_llm_cache(SQLiteCache(database_path='react_agent_cache.langchain.db'))

In [69]:
#initialsing the prompts
template = '''
    Answer this question as best as you can in a human way
    question : {question}
'''

prompt_template = PromptTemplate.from_template(template)
# prompt = hub.pull("hwchase17/react")

In [70]:
#initialising the Tools

#Python Repl Tool
py_repl = PythonREPLTool()
python_repl_tool = Tool(
    name = "Python Repl Tool",
    func = py_repl.run,
    description = "Use this tool to execute the Python codes and get results, You should input python code only"
)

#Wikipedia Tool
wikipedia_wrapper = WikipediaAPIWrapper(doc_content_chars_max=5000, lang="en", top_k_results="2")
wiki_init = WikipediaQueryRun(api_wrapper=wikipedia_wrapper)
wikipedia_tool = Tool(
    name="Wikipedia Tool",
    func = wiki_init.run,
    description = "Use this tools to get the latest information about a query from Wikipedia about a topic, country or person"
)

#DuckDuckGo Search Tool
duckduckgo_wrapper = DuckDuckGoSearchAPIWrapper(max_results=10)
duckduckgo_init = DuckDuckGoSearchRun(api_wrapper=duckduckgo_wrapper)
duckduckgo_tool = Tool(
    name="Duckduckgo Searh Tool",
    func = duckduckgo_init.run,
    description="Use this Tool to get latest information from Duckduckgo Search about a topic, person, or query that another tools can't provide"
)

In [71]:
tools= [python_repl_tool, wikipedia_tool, duckduckgo_tool]

In [72]:
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

In [73]:
question = "Find the first 30 number in a febonacci series"
output = agent_executor.invoke(
    {
        "input": prompt_template.format(question=question)
    }
)



> Entering new AgentExecutor chain...
Thought: The Fibonacci series is a sequence of numbers where each number is the sum of the two preceding ones, typically starting with 0 and 1. I will generate the first 30 numbers in the Fibonacci series.

Action: Python Repl Tool  
Action Input: 
```python
def fibonacci(n):
    series = [0, 1]
    while len(series) < n:
        series.append(series[-1] + series[-2])
    return series

fibonacci_30 = fibonacci(30)
fibonacci_30
```
Thought: I have generated the first 30 numbers in the Fibonacci series. I can now provide the sequence as the answer.

Final Answer: The first 30 numbers in the Fibonacci series are:  
0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418, 317811, 514229

> Finished chain.


In [74]:
question = "Who is Lala Lajpat Ray"
output = agent_executor.invoke(
    {
        "input": prompt_template.format(question=question)
    }
)



> Entering new AgentExecutor chain...
Thought: To provide an accurate and concise answer about who Lala Lajpat Ray was, it would be best to retrieve information from a reliable source like Wikipedia.

Action: Wikipedia Tool
Action Input: "Lala Lajpat Ray"
Page: Bipin Chandra Pal
Summary: Bipin Chandra Pal (Bengali: বিপিন চন্দ্র পাল ; 7 November 1858 – 20 May 1932) was an Indian nationalist, writer, orator, social reformer and freedom fighter. He was one third of the "Lal Bal Pal" triumvirate. He was one of the main architects of the Swadeshi movement. He is known as the Father of Revolutionary Thoughts in India. He also opposed the partition of Bengal by the British colonial government.

Page: List of Baniyas
Summary: This is a partial list of notable/famous people from Baniya community.Thought: The initial Wikipedia search returned information about Bipin Chandra Pal, who was part of the "Lal Bal Pal" trio along with Lala Lajpat Ray. To get specific details on Lala Lajpat Ray, I nee

In [30]:
question = "Who is user @c9hariom"
output = agent_executor.invoke(
    {
        "input": prompt_template.format(question=question)
    }
)



> Entering new AgentExecutor chain...


Impersonate 'firefox_133' does not exist, using 'random'


Thought: To answer who the user @c9hariom is, I need to search for information about this username. Since this looks like a social media or internet handle, it might not be something extensively covered on Wikipedia. I will use the Duckduckgo Search Tool to find any relevant recent information about the user @c9hariom.

Action: Duckduckgo Searh Tool
Action Input: "@c9hariom" user profile or information
If your Chrome profiles disappeared after using "End Task" in Task Manager, but you still see the old profile folders in C:\Users\My_Username\AppData\Local\Google\Chrome\User Data, you can try the following methods to restore them. The directory for Chrome profiles can be found under C:\\Users\\USERNAME\\AppData\\Local\\Google\\Chrome\\User Data, where various profiles are stored, including Default, Guest Profile , and Profile 1. If your data was synced with a Google account, recovery becomes simpler since the information is not entirely lost but rather exists on the cloud. A Chrome brow

In [77]:
llm.invoke("What is OpenAi")

AIMessage(content='OpenAI is an artificial intelligence research organization and company that develops advanced AI technologies with the goal of ensuring that artificial general intelligence (AGI) benefits all of humanity. Founded in December 2015, OpenAI focuses on creating safe and powerful AI models, conducting research in machine learning, natural language processing, and other AI domains. Some of its well-known projects include the GPT series of language models (like the one you’re interacting with), DALL·E for image generation, and Codex for code generation. OpenAI aims to promote and develop AI in a way that is safe, transparent, and broadly distributed.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 11, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'ca

Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: 
